# Central Park Node Workflow — Adaptive Enclosing Rectangle Version

This notebook is a **Central Park-only** workflow.

It will:
1. Download the **walkable path network** for Central Park from OpenStreetMap using **OSMnx**.
2. Download the **Central Park boundary** from OSM.
3. Build a graph from the walkable path lines.
4. Automatically identify candidate nodes.
5. Compute an **adaptive enclosing rectangle** for the park.
6. Use that rectangle as the basis for grid generation.
7. Assign **grid-based codes** to nodes based on the adaptive rectangle grid.
8. Display the boundary, adaptive rectangle, grid, grid labels, and nodes on an interactive map.
9. Export all outputs.

Example grid-based code:
- `11001` means: rectangle-grid cell `(x=1, y=1)`, node `001` inside that cell.

Notes:
- Here the enclosing shape is constrained to be a **rectangle (or square as a special case)**.
- The rectangle is the park hull’s **minimum rotated rectangle**, which is the most stable enclosing rectangle for this use case.


## Install dependencies

Run this in Terminal if needed:

```bash
pip install geopandas shapely networkx pandas numpy pyproj osmnx folium matplotlib mapclassify
```


In [1]:
from __future__ import annotations

import math
from pathlib import Path

import geopandas as gpd
import networkx as nx
import numpy as np
import pandas as pd
from shapely.geometry import GeometryCollection, LineString, MultiLineString, Point, Polygon
from shapely.ops import substring, unary_union


def angle_between(v1, v2):
    x1, y1 = v1
    x2, y2 = v2
    n1 = math.hypot(x1, y1)
    n2 = math.hypot(x2, y2)
    if n1 == 0 or n2 == 0:
        return 0.0
    dot = max(-1.0, min(1.0, (x1 * x2 + y1 * y2) / (n1 * n2)))
    return float(math.degrees(math.acos(dot)))

def straight_distance(line):
    coords = list(line.coords)
    if len(coords) < 2:
        return 0.0
    return float(Point(coords[0]).distance(Point(coords[-1])))

def line_sinuosity(line):
    direct = straight_distance(line)
    if direct <= 1e-9:
        return 1.0
    return float(line.length / direct)

def cumulative_heading_change(line):
    coords = list(line.coords)
    if len(coords) < 3:
        return 0.0
    total = 0.0
    for i in range(1, len(coords) - 1):
        p0, p1, p2 = coords[i - 1], coords[i], coords[i + 1]
        v1 = (p1[0] - p0[0], p1[1] - p0[1])
        v2 = (p2[0] - p1[0], p2[1] - p1[1])
        total += angle_between(v1, v2)
    return float(total)

def representative_curvature(line):
    if float(line.length) <= 1e-9:
        return 0.0
    return float(cumulative_heading_change(line) / float(line.length))

def flatten_lines(geom):
    if geom is None or geom.is_empty:
        return []
    if isinstance(geom, LineString):
        return [geom] if len(geom.coords) >= 2 else []
    if isinstance(geom, MultiLineString):
        out = []
        for g in geom.geoms:
            out.extend(flatten_lines(g))
        return out
    if isinstance(geom, GeometryCollection):
        out = []
        for g in geom.geoms:
            out.extend(flatten_lines(g))
        return out
    return []

def ensure_wgs84(gdf):
    if gdf.crs is None:
        gdf = gdf.set_crs('EPSG:4326')
    elif str(gdf.crs).upper() != 'EPSG:4326':
        gdf = gdf.to_crs('EPSG:4326')
    return gdf

def project_to_local_metric(gdf):
    gdf = ensure_wgs84(gdf)
    utm = gdf.estimate_utm_crs()
    if utm is None:
        raise ValueError('Could not estimate local metric CRS.')
    return gdf.to_crs(utm), str(utm)

def round_point_key(pt, precision=3):
    return (round(float(pt.x), precision), round(float(pt.y), precision))

def fetch_osmnx_walk_network(place_name, output_dir, stem):
    import osmnx as ox
    G = ox.graph_from_place(place_name, network_type='walk', simplify=True)
    nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)
    output_dir.mkdir(parents=True, exist_ok=True)
    nodes_gdf.to_crs('EPSG:4326').to_file(output_dir / f'{stem}_nodes.geojson', driver='GeoJSON')
    edges_gdf.to_crs('EPSG:4326').to_file(output_dir / f'{stem}_edges.geojson', driver='GeoJSON')
    return G, nodes_gdf, edges_gdf

def fetch_park_boundary(place_name, output_dir, stem):
    import osmnx as ox
    boundary_gdf = ox.geocode_to_gdf(place_name)
    boundary_gdf = ensure_wgs84(boundary_gdf)
    output_dir.mkdir(parents=True, exist_ok=True)
    boundary_gdf.to_file(output_dir / f'{stem}_boundary.geojson', driver='GeoJSON')
    return boundary_gdf

def build_graph_from_lines(lines_metric, node_round_precision=3):
    unioned = unary_union(list(lines_metric.geometry))
    segments = flatten_lines(unioned)
    G = nx.Graph()
    coord_to_node = {}
    next_node_num = 1
    next_edge_num = 1

    def get_node_id(pt):
        nonlocal next_node_num
        key = round_point_key(pt, precision=node_round_precision)
        if key not in coord_to_node:
            node_id = f'N{next_node_num:04d}'
            coord_to_node[key] = node_id
            G.add_node(node_id, geometry=Point(key[0], key[1]))
            next_node_num += 1
        return coord_to_node[key]

    for seg in segments:
        if seg.length <= 1e-6 or len(seg.coords) < 2:
            continue
        u = get_node_id(Point(seg.coords[0]))
        v = get_node_id(Point(seg.coords[-1]))
        edge_id = f'E{next_edge_num:05d}'
        next_edge_num += 1
        attrs = {
            'edge_id': edge_id,
            'geometry': seg,
            'length_m': float(seg.length),
            'sinuosity': line_sinuosity(seg),
            'cum_turn_deg': cumulative_heading_change(seg),
            'curvature': representative_curvature(seg),
            'near_road_crossing': False,
            'has_stairs': False,
            'has_signalized_crossing': False,
            'surface': 'unknown',
        }
        if G.has_edge(u, v):
            if attrs['length_m'] < G[u][v]['length_m']:
                G[u][v].update(attrs)
        else:
            G.add_edge(u, v, **attrs)
    return G

def graph_nodes_to_gdf(G, crs):
    rows = []
    for node_id, attrs in G.nodes(data=True):
        degree = int(G.degree(node_id))
        node_type = 'junction' if degree >= 3 else ('dead_end' if degree == 1 else 'through_node')
        rows.append({
            'node_id': node_id,
            'node_type': node_type,
            'node_subtype': None,
            'degree': degree,
            'source': 'graph',
            'source_edge': None,
            'name': node_id,
            'geometry': attrs['geometry'],
        })
    gdf = gpd.GeoDataFrame(rows, geometry='geometry', crs=crs)
    return gdf[gdf['node_type'].isin(['junction', 'dead_end'])].copy()

def intermediate_points_on_line(line, max_segment_len=80.0, max_turn_deg=45.0, min_spacing=20.0):
    out = []
    L = float(line.length)
    if L <= 1e-9:
        return out
    if L > max_segment_len:
        n_splits = int(L // max_segment_len)
        for i in range(1, n_splits + 1):
            d = min(i * max_segment_len, L - 1e-6)
            if 0 < d < L:
                out.append((line.interpolate(d), 'length_confirmation'))
    coords = list(line.coords)
    if len(coords) >= 3:
        accumulated_turn = 0.0
        last_added_distance = 0.0
        vertex_dist = [0.0]
        for i in range(1, len(coords)):
            vertex_dist.append(vertex_dist[-1] + Point(coords[i - 1]).distance(Point(coords[i])))
        for i in range(1, len(coords) - 1):
            p0, p1, p2 = coords[i - 1], coords[i], coords[i + 1]
            turn = angle_between((p1[0] - p0[0], p1[1] - p0[1]), (p2[0] - p1[0], p2[1] - p1[1]))
            accumulated_turn += abs(turn)
            d_here = vertex_dist[i]
            if accumulated_turn >= max_turn_deg and (d_here - last_added_distance) >= min_spacing:
                if min_spacing <= d_here <= max(L - min_spacing, min_spacing):
                    out.append((Point(p1), 'curve_confirmation'))
                    last_added_distance = d_here
                    accumulated_turn = 0.0
    return out

def extract_intermediate_candidates(G, crs, max_segment_len=80.0, max_turn_deg=45.0, min_spacing=20.0):
    rows = []
    idx = 1
    for _, _, data in G.edges(data=True):
        for pt, reason in intermediate_points_on_line(data['geometry'], max_segment_len=max_segment_len, max_turn_deg=max_turn_deg, min_spacing=min_spacing):
            rows.append({
                'node_id': f'C_INT_{idx:04d}',
                'node_type': 'intermediate_confirmation',
                'node_subtype': reason,
                'degree': None,
                'source': 'edge_rule',
                'source_edge': data['edge_id'],
                'name': f'{reason}_{idx}',
                'geometry': pt,
            })
            idx += 1
    return gpd.GeoDataFrame(rows, geometry='geometry', crs=crs)

def merge_close_candidate_nodes(structural_nodes, non_structural_nodes, merge_threshold=12.0):
    accepted_rows = []
    accepted_geoms = []
    for rec in structural_nodes.itertuples():
        accepted_rows.append({
            'node_id': rec.node_id,
            'node_type': rec.node_type,
            'node_subtype': rec.node_subtype,
            'degree': rec.degree,
            'source': rec.source,
            'source_edge': rec.source_edge,
            'name': rec.name,
            'notes': '',
            'geometry': rec.geometry,
        })
        accepted_geoms.append(rec.geometry)
    for rec in non_structural_nodes.itertuples():
        nearest_idx = None
        nearest_dist = None
        for i, g in enumerate(accepted_geoms):
            d = float(g.distance(rec.geometry))
            if nearest_dist is None or d < nearest_dist:
                nearest_idx = i
                nearest_dist = d
        if nearest_idx is not None and nearest_dist is not None and nearest_dist <= merge_threshold:
            tag = rec.node_subtype or rec.node_type
            if accepted_rows[nearest_idx]['notes']:
                if tag not in accepted_rows[nearest_idx]['notes']:
                    accepted_rows[nearest_idx]['notes'] += f'; {tag}'
            else:
                accepted_rows[nearest_idx]['notes'] = tag
        else:
            accepted_rows.append({
                'node_id': rec.node_id,
                'node_type': rec.node_type,
                'node_subtype': rec.node_subtype,
                'degree': rec.degree,
                'source': rec.source,
                'source_edge': rec.source_edge,
                'name': rec.name,
                'notes': '',
                'geometry': rec.geometry,
            })
            accepted_geoms.append(rec.geometry)
    return gpd.GeoDataFrame(accepted_rows, geometry='geometry', crs=structural_nodes.crs)

def make_adaptive_enclosing_rectangle(boundary_gdf):
    boundary_gdf = ensure_wgs84(boundary_gdf)
    boundary_metric, metric_crs = project_to_local_metric(boundary_gdf)
    hull_poly = unary_union(list(boundary_metric.geometry)).convex_hull
    rect_poly = hull_poly.minimum_rotated_rectangle
    rect_gdf_metric = gpd.GeoDataFrame(
        [{'geometry': rect_poly, 'rect_area': rect_poly.area, 'hull_area': hull_poly.area}],
        geometry='geometry',
        crs=metric_crs
    )
    rect_gdf = rect_gdf_metric.to_crs('EPSG:4326')
    extra_area = float(rect_poly.area - hull_poly.area)
    return rect_gdf, hull_poly, metric_crs, extra_area

def order_rect_corners(rect_poly_metric, hull_poly_metric):
    pts = np.asarray(list(rect_poly_metric.exterior.coords)[:-1], dtype=float)
    center = np.array(rect_poly_metric.centroid.coords[0], dtype=float)
    hull_coords = np.asarray(hull_poly_metric.exterior.coords[:-1], dtype=float)
    arr = hull_coords - center
    cov = np.cov(arr.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    major = eigvecs[:, np.argmax(eigvals)]
    minor = eigvecs[:, np.argmin(eigvals)]
    proj_major = (pts - center) @ major
    proj_minor = (pts - center) @ minor
    lower_idx = np.argsort(proj_minor)[:2]
    upper_idx = np.argsort(proj_minor)[2:]
    lower_idx = lower_idx[np.argsort(proj_major[lower_idx])]
    upper_idx = upper_idx[np.argsort(proj_major[upper_idx])]
    sw = pts[lower_idx[0]]
    se = pts[lower_idx[1]]
    nw = pts[upper_idx[0]]
    ne = pts[upper_idx[1]]
    return np.array([sw, se, ne, nw], dtype=float)

def _bilinear_rect_point(sw, se, ne, nw, s, t):
    return (
        (1 - s) * (1 - t) * sw
        + s * (1 - t) * se
        + s * t * ne
        + (1 - s) * t * nw
    )

def make_grid_from_rectangle(rect_gdf, hull_poly_metric, metric_crs, n_cols=5, n_rows=5):
    rect_metric = rect_gdf.to_crs(metric_crs)
    rect_poly_metric = rect_metric.geometry.iloc[0]
    sw, se, ne, nw = order_rect_corners(rect_poly_metric, hull_poly_metric)
    cells = []
    for ix in range(n_cols):
        for iy in range(n_rows):
            s0 = ix / n_cols
            s1 = (ix + 1) / n_cols
            t0 = iy / n_rows
            t1 = (iy + 1) / n_rows
            p00 = _bilinear_rect_point(sw, se, ne, nw, s0, t0)
            p10 = _bilinear_rect_point(sw, se, ne, nw, s1, t0)
            p11 = _bilinear_rect_point(sw, se, ne, nw, s1, t1)
            p01 = _bilinear_rect_point(sw, se, ne, nw, s0, t1)
            cell_poly = Polygon([tuple(p00), tuple(p10), tuple(p11), tuple(p01)])
            cells.append({'grid_x': ix + 1, 'grid_y': iy + 1, 'grid_id': f'{ix + 1}{iy + 1}', 'geometry': cell_poly})
    grid_metric = gpd.GeoDataFrame(cells, geometry='geometry', crs=metric_crs)
    return grid_metric.to_crs('EPSG:4326')

def assign_grid_codes_from_grid(nodes_gdf, grid_gdf, park_prefix=None):
    nodes = ensure_wgs84(nodes_gdf).copy()
    grid = ensure_wgs84(grid_gdf).copy()
    joined = gpd.sjoin(nodes, grid[['grid_x', 'grid_y', 'grid_id', 'geometry']], how='left', predicate='within')
    missing = joined['grid_x'].isna()
    if missing.any():
        nearest = gpd.sjoin_nearest(nodes.loc[missing].copy(), grid[['grid_x', 'grid_y', 'grid_id', 'geometry']], how='left', distance_col='grid_join_dist')
        for col in ['grid_x', 'grid_y', 'grid_id']:
            joined.loc[missing, col] = nearest[col].values
    joined['lat'] = joined.geometry.y
    joined['lon'] = joined.geometry.x
    joined = joined.sort_values(by=['grid_x', 'grid_y', 'lat', 'lon'], ascending=[True, True, False, True]).reset_index(drop=True)
    joined['cell_seq'] = joined.groupby(['grid_x', 'grid_y']).cumcount() + 1
    def make_code(row):
        xy = f"{int(row['grid_x'])}{int(row['grid_y'])}"
        seq = f"{int(row['cell_seq']):03d}"
        if park_prefix:
            return f"{park_prefix}-{xy}-{seq}"
        return f"{xy}{seq}"
    joined['grid_node_code'] = joined.apply(make_code, axis=1)
    return joined

def build_augmented_graph(base_G, final_nodes):
    extra_by_edge = {}
    base_edge_lookup = {}
    for u, v, data in base_G.edges(data=True):
        base_edge_lookup[data['edge_id']] = (u, v, data)
        extra_by_edge[data['edge_id']] = []
    for rec in final_nodes.itertuples():
        if rec.source_edge is None or rec.source_edge not in base_edge_lookup:
            continue
        u, v, data = base_edge_lookup[rec.source_edge]
        line = data['geometry']
        d = float(line.project(rec.geometry))
        if d <= 1e-6 or d >= line.length - 1e-6:
            continue
        if float(line.distance(rec.geometry)) > 2.0:
            continue
        extra_by_edge[rec.source_edge].append((d, rec.node_id))
    G2 = nx.Graph()
    for rec in final_nodes.itertuples():
        G2.add_node(rec.node_id, geometry=rec.geometry, node_type=rec.node_type, node_subtype=rec.node_subtype, name=rec.name, notes=getattr(rec, 'notes', ''))
    new_edge_num = 1
    for edge_id, (u, v, data) in base_edge_lookup.items():
        line = data['geometry']
        extra_nodes = sorted(extra_by_edge.get(edge_id, []), key=lambda x: x[0])
        distances = [0.0] + [d for d, _ in extra_nodes] + [float(line.length)]
        node_ids = [u] + [nid for _, nid in extra_nodes] + [v]
        for i in range(len(node_ids) - 1):
            d0, d1 = distances[i], distances[i + 1]
            if d1 - d0 <= 1e-6:
                continue
            seg = substring(line, d0, d1)
            if seg.is_empty or not isinstance(seg, LineString) or len(seg.coords) < 2:
                continue
            G2.add_edge(
                node_ids[i], node_ids[i + 1],
                edge_id=f'AE{new_edge_num:05d}',
                parent_edge_id=edge_id,
                geometry=seg,
                length_m=float(seg.length),
                sinuosity=line_sinuosity(seg),
                cum_turn_deg=cumulative_heading_change(seg),
                curvature=representative_curvature(seg),
                near_road_crossing=data.get('near_road_crossing', False),
                has_stairs=data.get('has_stairs', False),
                has_signalized_crossing=data.get('has_signalized_crossing', False),
                surface=data.get('surface', 'unknown'),
            )
            new_edge_num += 1
    return G2

def graph_edges_to_gdf(G, crs):
    rows = []
    for u, v, data in G.edges(data=True):
        rows.append({
            'u': u, 'v': v, 'edge_id': data['edge_id'], 'length_m': data['length_m'],
            'sinuosity': data['sinuosity'], 'cum_turn_deg': data['cum_turn_deg'],
            'curvature': data['curvature'], 'near_road_crossing': data.get('near_road_crossing', False),
            'has_stairs': data.get('has_stairs', False), 'has_signalized_crossing': data.get('has_signalized_crossing', False),
            'surface': data.get('surface', 'unknown'), 'geometry': data['geometry'],
        })
    return gpd.GeoDataFrame(rows, geometry='geometry', crs=crs)

def export_outputs(output_dir, base_nodes_metric, final_nodes_metric, base_edges_metric, augmented_edges_metric):
    output_dir.mkdir(parents=True, exist_ok=True)
    base_nodes_metric.to_crs('EPSG:4326').to_file(output_dir / 'base_structural_nodes.geojson', driver='GeoJSON')
    final_nodes_metric.to_crs('EPSG:4326').to_file(output_dir / 'final_candidate_nodes.geojson', driver='GeoJSON')
    base_edges_metric.to_crs('EPSG:4326').to_file(output_dir / 'base_graph_edges.geojson', driver='GeoJSON')
    augmented_edges_metric.to_crs('EPSG:4326').to_file(output_dir / 'augmented_graph_edges.geojson', driver='GeoJSON')
    final_nodes_df = final_nodes_metric.to_crs('EPSG:4326').copy()
    final_nodes_df['lat'] = final_nodes_df.geometry.y
    final_nodes_df['lon'] = final_nodes_df.geometry.x
    final_nodes_df.drop(columns='geometry').to_csv(output_dir / 'final_candidate_nodes.csv', index=False)


## 1. Download the Central Park walkable network and boundary


In [2]:
# If needed, uncomment and run once:
# !pip install osmnx

from pathlib import Path

place_name = 'Central Park, Manhattan, New York, USA'
park_dir = Path('central_osmnx_outputs')

G_osm, osm_nodes_gdf, osm_edges_gdf = fetch_osmnx_walk_network(place_name, park_dir, 'central_park')
boundary_gdf = fetch_park_boundary(place_name, park_dir, 'central_park')

print('Downloaded Central Park walk network and boundary.')
print('Nodes:', G_osm.number_of_nodes())
print('Edges:', G_osm.number_of_edges())
print('Saved edges to:', park_dir / 'central_park_edges.geojson')
print('Saved boundary to:', park_dir / 'central_park_boundary.geojson')


Downloaded Central Park walk network and boundary.
Nodes: 1412
Edges: 4170
Saved edges to: central_osmnx_outputs/central_park_edges.geojson
Saved boundary to: central_osmnx_outputs/central_park_boundary.geojson


## 2. Inspect the walkable path file and boundary


In [3]:
from pathlib import Path
import geopandas as gpd

edges_path = Path('central_osmnx_outputs/central_park_edges.geojson')
boundary_path = Path('central_osmnx_outputs/central_park_boundary.geojson')

edges_raw = gpd.read_file(edges_path)
boundary_raw = gpd.read_file(boundary_path)

print('Edges geometry types:')
print(edges_raw.geometry.geom_type.value_counts(dropna=False))
print('\nBoundary geometry types:')
print(boundary_raw.geometry.geom_type.value_counts(dropna=False))


Skipping field highway: unsupported OGR type: 5
Skipping field name: unsupported OGR type: 5
Skipping field lanes: unsupported OGR type: 5


Edges geometry types:
LineString    4170
Name: count, dtype: int64

Boundary geometry types:
Polygon    1
Name: count, dtype: int64


## 3. Run node identification for Central Park


In [4]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

input_path = Path('central_osmnx_outputs/central_park_edges.geojson')
output_dir = Path('central_outputs')

raw = gpd.read_file(input_path)
if raw.crs is None:
    raw = raw.set_crs('EPSG:4326')
elif str(raw.crs).upper() != 'EPSG:4326':
    raw = raw.to_crs('EPSG:4326')

raw = raw[raw.geometry.notna()].copy()
raw = raw[~raw.geometry.is_empty].copy()

lines_wgs = raw[raw.geometry.geom_type.isin(['LineString', 'MultiLineString'])].copy()
lines_metric, metric_crs = project_to_local_metric(lines_wgs)

base_G = build_graph_from_lines(lines_metric)
structural_nodes = graph_nodes_to_gdf(base_G, metric_crs)

intermediate_nodes = extract_intermediate_candidates(
    base_G,
    crs=metric_crs,
    max_segment_len=80.0,
    max_turn_deg=45.0,
    min_spacing=20.0,
)

non_structural = gpd.GeoDataFrame(
    pd.concat([intermediate_nodes], ignore_index=True),
    geometry='geometry',
    crs=metric_crs,
)

final_nodes_metric = merge_close_candidate_nodes(
    structural_nodes,
    non_structural,
    merge_threshold=12.0,
)

augmented_G = build_augmented_graph(base_G, final_nodes_metric)
base_edges_metric = graph_edges_to_gdf(base_G, metric_crs)
augmented_edges_metric = graph_edges_to_gdf(augmented_G, metric_crs)

export_outputs(
    output_dir,
    structural_nodes,
    final_nodes_metric,
    base_edges_metric,
    augmented_edges_metric,
)

print('Done.')
print('Metric CRS:', metric_crs)
print('Base graph nodes:', base_G.number_of_nodes())
print('Base graph edges:', base_G.number_of_edges())
print('Structural nodes:', len(structural_nodes))
print('Intermediate nodes:', len(intermediate_nodes))
print('Final candidate nodes:', len(final_nodes_metric))


Skipping field highway: unsupported OGR type: 5
Skipping field name: unsupported OGR type: 5
Skipping field lanes: unsupported OGR type: 5


Done.
Metric CRS: EPSG:32618
Base graph nodes: 13580
Base graph edges: 14300
Structural nodes: 1401
Intermediate nodes: 21
Final candidate nodes: 1418


## Add useful infrastructure nodes snapped to walkable roads

This section downloads a curated set of navigation-relevant park infrastructure from OSM, converts each feature to a representative point, snaps that point onto the nearest walkable road, and adds those snapped locations as **green infrastructure nodes**. The combined node layer can then be exported, coded, visualized, and routed on the augmented graph.


In [5]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, LineString, MultiLineString, Polygon, MultiPolygon, GeometryCollection

# Preserve the original node layer as the core node set
core_candidate_nodes_metric = final_nodes_metric.copy()
core_candidate_nodes_metric['display_group'] = 'core'
core_candidate_nodes_metric['display_color'] = 'red'
core_candidate_nodes_metric['display_label'] = 'Core candidate node'

INFRA_CATEGORY_STYLES = {
    'Restroom': '#1f77b4',
    'Drinking water': '#17becf',
    'Shelter': '#8c564b',
    'Food / cafe': '#ff7f0e',
    'First aid': '#d62728',
    'Picnic': '#2ca02c',
    'Information / landmark': '#9467bd',
    'Recreation': '#e377c2',
    'Entrance / gate': '#7f7f7f',
    'Other': '#bcbd22',
}

def geometry_to_point(geom):
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type == 'Point':
        return geom
    if geom.geom_type in ['Polygon', 'MultiPolygon', 'LineString', 'MultiLineString']:
        return geom.representative_point()
    if geom.geom_type == 'GeometryCollection':
        for sub in geom.geoms:
            pt = geometry_to_point(sub)
            if pt is not None:
                return pt
        return None
    try:
        return geom.representative_point()
    except Exception:
        return None

def pick_infra_kind(row):
    checks = [
        ('amenity', ['toilets', 'drinking_water', 'shelter', 'cafe', 'restaurant', 'first_aid', 'picnic_table']),
        ('tourism', ['information', 'viewpoint', 'attraction', 'museum', 'artwork']),
        ('leisure', ['playground', 'sports_centre', 'pitch']),
        ('barrier', ['gate']),
    ]
    for field, allowed in checks:
        value = row.get(field)
        if pd.notna(value) and value in allowed:
            return field, str(value)
    entrance_value = row.get('entrance')
    if pd.notna(entrance_value):
        return 'entrance', str(entrance_value)
    return 'other', 'other'

def classify_infra(field, infra_type):
    if infra_type == 'toilets':
        return 'Restroom'
    if infra_type == 'drinking_water':
        return 'Drinking water'
    if infra_type == 'shelter':
        return 'Shelter'
    if infra_type in ['cafe', 'restaurant']:
        return 'Food / cafe'
    if infra_type == 'first_aid':
        return 'First aid'
    if infra_type == 'picnic_table':
        return 'Picnic'
    if field == 'tourism':
        return 'Information / landmark'
    if field == 'leisure':
        return 'Recreation'
    if field in ['entrance', 'barrier']:
        return 'Entrance / gate'
    return 'Other'

def is_gate_like(field, infra_type):
    return field in ['entrance', 'barrier'] or infra_type == 'gate'

def extract_gate_name(row):
    # Prefer explicit naming fields so the exported gate label can be accurate
    for key in ['name', 'official_name', 'alt_name', 'short_name', 'ref']:
        value = row.get(key)
        if pd.notna(value) and str(value).strip():
            return str(value).strip()
    return None

def make_infra_label(row, fallback_type, infra_field=None):
    gate_name = extract_gate_name(row) if is_gate_like(infra_field, fallback_type) else None
    if gate_name:
        return gate_name

    for key in ['name', 'official_name', 'operator', 'brand']:
        value = row.get(key)
        if pd.notna(value) and str(value).strip():
            return str(value).strip()

    if infra_field == 'entrance':
        entrance_value = row.get('entrance')
        if pd.notna(entrance_value) and str(entrance_value).strip():
            return f"Unnamed {str(entrance_value).replace('_', ' ')} entrance"

    if fallback_type == 'gate' or infra_field == 'barrier':
        return 'Unnamed gate'

    return fallback_type.replace('_', ' ').title()

park_boundary_wgs = ensure_wgs84(boundary_gdf).copy()
park_polygon = park_boundary_wgs.geometry.union_all() if hasattr(park_boundary_wgs.geometry, 'union_all') else park_boundary_wgs.unary_union
if isinstance(park_polygon, MultiPolygon):
    park_polygon = max(list(park_polygon.geoms), key=lambda g: g.area)

useful_infra_tags = {
    'amenity': ['toilets', 'drinking_water', 'shelter', 'cafe', 'restaurant', 'first_aid', 'picnic_table'],
    'tourism': ['information', 'viewpoint', 'attraction', 'museum', 'artwork'],
    'leisure': ['playground', 'sports_centre', 'pitch'],
    'entrance': True,
    'barrier': ['gate'],
}

infra_raw = ox.features_from_polygon(park_polygon, tags=useful_infra_tags).reset_index()
infra_raw = infra_raw[infra_raw.geometry.notna()].copy()
infra_raw = infra_raw[~infra_raw.geometry.is_empty].copy()

infra_points = infra_raw.copy()
infra_points['geometry'] = infra_points.geometry.apply(geometry_to_point)
infra_points = infra_points[infra_points.geometry.notna()].copy()
infra_points = gpd.GeoDataFrame(infra_points, geometry='geometry', crs='EPSG:4326')
infra_points_metric = infra_points.to_crs(metric_crs).copy()

snapped_rows = []
for idx, row in infra_points_metric.iterrows():
    infra_pt = row.geometry
    dists = base_edges_metric.geometry.distance(infra_pt)
    nearest_edge_idx = dists.idxmin()
    nearest_edge = base_edges_metric.loc[nearest_edge_idx]
    nearest_line = nearest_edge.geometry
    proj_dist = float(nearest_line.project(infra_pt))
    snapped_pt = nearest_line.interpolate(proj_dist)

    infra_field, infra_type = pick_infra_kind(row)
    infra_class = classify_infra(infra_field, infra_type)
    gate_name = extract_gate_name(row) if is_gate_like(infra_field, infra_type) else None
    infra_label = make_infra_label(row, infra_type, infra_field=infra_field)
    snap_dist = float(infra_pt.distance(snapped_pt))

    snapped_rows.append({
        'node_id': f'INF_{len(snapped_rows) + 1:04d}',
        'node_type': 'infrastructure',
        'node_subtype': f'{infra_field}:{infra_type}',
        'degree': None,
        'source': 'infrastructure',
        'source_edge': nearest_edge.get('edge_id', None),
        'name': gate_name if gate_name else infra_label,
        'notes': f'useful_infrastructure; {infra_field}:{infra_type}',
        'infra_field': infra_field,
        'infra_type': infra_type,
        'infra_class': infra_class,
        'infra_category': infra_class,
        'infra_name': infra_label,
        'display_name': gate_name if gate_name else infra_label,
        'gate_name': gate_name,
        'gate_name_status': (
            'named' if gate_name else ('missing_name' if is_gate_like(infra_field, infra_type) else None)
        ),
        'needs_gate_name_review': bool(is_gate_like(infra_field, infra_type) and not gate_name),
        'snap_dist_m': snap_dist,
        'display_group': 'infrastructure',
        'display_label': infra_class,
        'display_color': INFRA_CATEGORY_STYLES.get(infra_class, '#bcbd22'),
        'geometry': snapped_pt,
    })

infra_nodes_metric = gpd.GeoDataFrame(snapped_rows, geometry='geometry', crs=metric_crs)

# Keep only one snapped node per infrastructure name/type/location at ~1 m precision
infra_nodes_metric['snap_x_round'] = infra_nodes_metric.geometry.x.round(0)
infra_nodes_metric['snap_y_round'] = infra_nodes_metric.geometry.y.round(0)
infra_nodes_metric = (
    infra_nodes_metric
    .sort_values(by=['infra_name', 'infra_type', 'snap_dist_m'])
    .drop_duplicates(subset=['infra_name', 'infra_type', 'snap_x_round', 'snap_y_round'])
    .drop(columns=['snap_x_round', 'snap_y_round'])
    .reset_index(drop=True)
)
infra_nodes_metric['node_id'] = [f'INF_{i:04d}' for i in range(1, len(infra_nodes_metric) + 1)]

# Combine the original node layer with the new infrastructure nodes
final_nodes_metric = gpd.GeoDataFrame(
    pd.concat([core_candidate_nodes_metric, infra_nodes_metric], ignore_index=True),
    geometry='geometry',
    crs=metric_crs,
)

augmented_G = build_augmented_graph(base_G, final_nodes_metric)
augmented_edges_metric = graph_edges_to_gdf(augmented_G, metric_crs)

# Overwrite exports so downstream files include the infrastructure nodes as part of the candidate layer
export_outputs(
    output_dir,
    structural_nodes,
    final_nodes_metric,
    base_edges_metric,
    augmented_edges_metric,
)

infra_nodes_export = infra_nodes_metric.to_crs('EPSG:4326').copy()
infra_nodes_export['lat'] = infra_nodes_export.geometry.y
infra_nodes_export['lon'] = infra_nodes_export.geometry.x
infra_nodes_export.to_file(output_dir / 'infrastructure_nodes_snapped.geojson', driver='GeoJSON')
infra_nodes_export.drop(columns='geometry').to_csv(output_dir / 'infrastructure_nodes_snapped.csv', index=False)

print('Useful infrastructure nodes added on walkable roads.')
print('Core candidate nodes:', len(core_candidate_nodes_metric))
print('Infrastructure nodes snapped to walkable roads:', len(infra_nodes_metric))
print('Combined candidate nodes:', len(final_nodes_metric))
print('Named / reviewed gate nodes:', int((infra_nodes_metric['gate_name_status'] == 'named').sum()))
print('Gate nodes still missing names:', int(infra_nodes_metric['needs_gate_name_review'].fillna(False).sum()))
print('Updated augmented graph nodes:', augmented_G.number_of_nodes())
print('Updated augmented graph edges:', augmented_G.number_of_edges())

/var/folders/r5/hpjs_bz54311tjr7rf9_dlwc0000gn/T/ipykernel_47507/1907003919.py:193: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat([core_candidate_nodes_metric, infra_nodes_metric], ignore_index=True),


Useful infrastructure nodes added on walkable roads.
Core candidate nodes: 1418
Infrastructure nodes snapped to walkable roads: 349
Combined candidate nodes: 1767
Named / reviewed gate nodes: 4
Gate nodes still missing names: 29
Updated augmented graph nodes: 13946
Updated augmented graph edges: 14569


## 4. Compute the adaptive enclosing rectangle


In [6]:
boundary_gdf = gpd.read_file('central_osmnx_outputs/central_park_boundary.geojson')
boundary_gdf = ensure_wgs84(boundary_gdf)

adaptive_rect_gdf, hull_poly_metric, rect_metric_crs, extra_area_metric = make_adaptive_enclosing_rectangle(boundary_gdf)

print('Adaptive rectangle computed.')
print('Extra area outside the park hull (square meters, approx):', extra_area_metric)

adaptive_rect_gdf


Adaptive rectangle computed.
Extra area outside the park hull (square meters, approx): 19830.65005086176


,geometry,rect_area,hull_area
0,"POLYGON ((-73.98169 40.76809, -73.95808 40.800...",3.452292e+06,3.432461e+06


## 5. Build a grid from the adaptive rectangle


In [7]:
# Change grid resolution here if needed
rect_grid_gdf = make_grid_from_rectangle(
    adaptive_rect_gdf,
    hull_poly_metric,
    rect_metric_crs,
    n_cols=5,
    n_rows=5
)

rect_grid_gdf.head()


,grid_x,grid_y,grid_id,geometry
0,1,1,11,"POLYGON ((-73.97299 40.76442, -73.96827 40.770..."
1,1,2,12,"POLYGON ((-73.97473 40.76516, -73.97001 40.771..."
2,1,3,13,"POLYGON ((-73.97647 40.76589, -73.97175 40.772..."
3,1,4,14,"POLYGON ((-73.97821 40.76662, -73.97349 40.773..."
4,1,5,15,"POLYGON ((-73.97995 40.76736, -73.97523 40.773..."


## 6. Assign grid-based node codes using the adaptive rectangle grid


In [8]:
grid_coded_nodes = assign_grid_codes_from_grid(final_nodes_metric, rect_grid_gdf, park_prefix=None)

print('Grid-coded nodes created.')
print(grid_coded_nodes[['node_id', 'grid_x', 'grid_y', 'cell_seq', 'grid_node_code']].head(20))


Grid-coded nodes created.
     node_id  grid_x  grid_y  cell_seq grid_node_code
0      N8478       1       1         1          11001
1      N3125       1       1         2          11002
2      N3124       1       1         3          11003
3      N1479       1       1         4          11004
4   INF_0094       1       1         5          11005
5      N1489       1       1         6          11006
6      N1484       1       1         7          11007
7      N1466       1       1         8          11008
8      N1441       1       1         9          11009
9   INF_0080       1       1        10          11010
10  INF_0261       1       1        11          11011
11     N1520       1       1        12          11012
12     N1385       1       1        13          11013
13  INF_0011       1       1        14          11014
14     N1368       1       1        15          11015
15     N1518       1       1        16          11016
16  INF_0014       1       1        17          11017
17

## 7. Export the adaptive rectangle, rectangle grid, and grid-coded node table


In [9]:
grid_output_dir = Path('central_outputs_gridcoded')
grid_output_dir.mkdir(parents=True, exist_ok=True)

adaptive_rect_gdf.to_file(grid_output_dir / 'adaptive_rectangle.geojson', driver='GeoJSON')
rect_grid_gdf.to_file(grid_output_dir / 'adaptive_rectangle_grid.geojson', driver='GeoJSON')

grid_nodes_export = grid_coded_nodes.copy()
grid_nodes_export.to_file(grid_output_dir / 'final_candidate_nodes_gridcoded.geojson', driver='GeoJSON')
grid_nodes_export.drop(columns='geometry').to_csv(grid_output_dir / 'final_candidate_nodes_gridcoded.csv', index=False)

if 'display_group' in grid_nodes_export.columns:
    infra_grid_nodes_export = grid_nodes_export[grid_nodes_export['display_group'] == 'infrastructure'].copy()
    if len(infra_grid_nodes_export) > 0:
        infra_grid_nodes_export.to_file(grid_output_dir / 'infrastructure_nodes_gridcoded.geojson', driver='GeoJSON')
        infra_grid_nodes_export.drop(columns='geometry').to_csv(grid_output_dir / 'infrastructure_nodes_gridcoded.csv', index=False)

print('Saved adaptive rectangle outputs.')
print('Saved files in:', grid_output_dir.resolve())

Saved adaptive rectangle outputs.
Saved files in: /Users/tianshushi/Desktop/park_navigation_full_package/notebooks/central_outputs_gridcoded


## 12. Export app-ready files for the web app prototype

In [10]:
from pathlib import Path
import json
import pickle
import geopandas as gpd
import pandas as pd

app_output_dir = Path('app_data')
app_output_dir.mkdir(parents=True, exist_ok=True)

# --- 1. Grid-coded candidate nodes ---
nodes_app = grid_coded_nodes.copy()
if nodes_app.crs is None or str(nodes_app.crs).upper() != 'EPSG:4326':
    nodes_app = ensure_wgs84(nodes_app)

nodes_app['lat'] = nodes_app.geometry.y
nodes_app['lon'] = nodes_app.geometry.x

if 'display_name' not in nodes_app.columns:
    nodes_app['display_name'] = nodes_app['name']

nodes_app.to_file(app_output_dir / 'final_candidate_nodes_gridcoded.geojson', driver='GeoJSON')
nodes_app.drop(columns='geometry').to_csv(app_output_dir / 'final_candidate_nodes_gridcoded.csv', index=False)

# --- 2. Infrastructure nodes subset ---
if 'display_group' in nodes_app.columns:
    infra_app = nodes_app[nodes_app['display_group'] == 'infrastructure'].copy()
else:
    infra_app = nodes_app[nodes_app['node_type'] == 'infrastructure'].copy()

if len(infra_app) > 0:
    infra_app.to_file(app_output_dir / 'infrastructure_nodes_gridcoded.geojson', driver='GeoJSON')
    infra_app.drop(columns='geometry').to_csv(app_output_dir / 'infrastructure_nodes_gridcoded.csv', index=False)

    gate_app = infra_app[infra_app['infra_class'] == 'Entrance / gate'].copy() if 'infra_class' in infra_app.columns else infra_app.iloc[0:0].copy()
    if len(gate_app) > 0:
        gate_app.to_file(app_output_dir / 'gate_nodes_gridcoded.geojson', driver='GeoJSON')
        gate_app.drop(columns='geometry').to_csv(app_output_dir / 'gate_nodes_gridcoded.csv', index=False)

# --- 3. Augmented edges layer for the app ---
augmented_edges_wgs = ensure_wgs84(augmented_edges_metric.copy())
augmented_edges_wgs.to_file(app_output_dir / 'augmented_graph_edges.geojson', driver='GeoJSON')

# --- 4. Optional graph pickle for direct loading in the app ---
with open(app_output_dir / 'park_graph.pkl', 'wb') as f:
    pickle.dump(augmented_G, f)

# --- 5. Small manifest for the app layer ---
manifest = {
    'park': 'Central Park',
    'nodes_csv': 'final_candidate_nodes_gridcoded.csv',
    'infra_csv': 'infrastructure_nodes_gridcoded.csv',
    'gate_csv': 'gate_nodes_gridcoded.csv',
    'edges_geojson': 'augmented_graph_edges.geojson',
    'graph_pickle': 'park_graph.pkl',
    'notes': 'Use these files as the app_data source for the navigation web app prototype.',
}
with open(app_output_dir / 'app_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('Saved app-ready files in:', app_output_dir.resolve())
for p in sorted(app_output_dir.iterdir()):
    print('-', p.name)

Saved app-ready files in: /Users/tianshushi/Desktop/park_navigation_full_package/notebooks/app_data
- app_manifest.json
- augmented_graph_edges.geojson
- final_candidate_nodes_gridcoded.csv
- final_candidate_nodes_gridcoded.geojson
- gate_nodes_gridcoded.csv
- gate_nodes_gridcoded.geojson
- infrastructure_nodes_gridcoded.csv
- infrastructure_nodes_gridcoded.geojson
- park_graph.pkl


## 8. Interactive map: boundary + adaptive rectangle + grid + nodes


In [11]:
# If explore() complains about missing packages, run:
# !pip install folium matplotlib mapclassify

import folium
from branca.element import MacroElement, Template

def add_infra_legend(m, color_map, title='Infrastructure nodes'):
    legend_items = ''.join([
        f"<div style='display:flex; align-items:center; margin-bottom:4px;'>"
        f"<span style='display:inline-block; width:12px; height:12px; background:{color}; "
        f"border:1px solid #333; margin-right:8px;'></span>{label}</div>"
        for label, color in color_map.items()
    ])
    html = f"""
    {{% macro html(this, kwargs) %}}
    <div style="
        position: fixed;
        bottom: 40px;
        left: 40px;
        width: 190px;
        z-index: 9999;
        background: white;
        border: 2px solid rgba(0,0,0,0.2);
        border-radius: 6px;
        padding: 10px 12px;
        font-size: 12px;
        box-shadow: 0 1px 5px rgba(0,0,0,0.3);
    ">
        <div style="font-weight: bold; margin-bottom: 8px;">{title}</div>
        <div style='display:flex; align-items:center; margin-bottom:6px;'>
            <span style='display:inline-block; width:12px; height:12px; background:red; border:1px solid #333; margin-right:8px;'></span>Core candidate node
        </div>
        {legend_items}
    </div>
    {{% endmacro %}}
    """
    macro = MacroElement()
    macro._template = Template(html)
    m.get_root().add_child(macro)
    return m

# Prefer the grid-coded layer for visualization when available
if 'grid_coded_nodes' in globals():
    display_nodes_metric = grid_coded_nodes.copy()
else:
    display_nodes_metric = final_nodes_metric.copy()

final_nodes_wgs = display_nodes_metric.to_crs('EPSG:4326').copy()
if 'display_group' in final_nodes_wgs.columns:
    core_nodes_wgs = final_nodes_wgs[final_nodes_wgs['display_group'] != 'infrastructure'].copy()
    infra_nodes_wgs = final_nodes_wgs[final_nodes_wgs['display_group'] == 'infrastructure'].copy()
else:
    core_nodes_wgs = final_nodes_wgs.copy()
    infra_nodes_wgs = final_nodes_wgs.iloc[0:0].copy()

m = boundary_gdf.explore(
    style_kwds={'fillOpacity': 0.02, 'color': 'black', 'weight': 2},
    tooltip=[]
)

adaptive_rect_gdf.explore(
    m=m,
    style_kwds={'fillOpacity': 0.03, 'color': 'green', 'weight': 2},
    tooltip=[]
)

rect_grid_gdf.explore(
    m=m,
    style_kwds={'fillOpacity': 0.03, 'color': 'blue', 'weight': 1},
    tooltip=['grid_id', 'grid_x', 'grid_y']
)

if len(core_nodes_wgs) > 0:
    core_tooltips = [
        col for col in ['grid_node_code', 'grid_x', 'grid_y', 'cell_seq', 'node_type', 'node_subtype']
        if col in core_nodes_wgs.columns
    ]
    core_nodes_wgs.explore(
        m=m,
        color='red',
        markersize=4,
        tooltip=core_tooltips
    )

if len(infra_nodes_wgs) > 0:
    infra_tooltip_cols = [c for c in ['node_id', 'infra_name', 'infra_class'] if c in infra_nodes_wgs.columns]
    infra_popup_cols = [c for c in [
        'node_id', 'grid_node_code', 'infra_name', 'infra_class', 'infra_type',
        'node_subtype', 'snap_dist_m', 'grid_x', 'grid_y'
    ] if c in infra_nodes_wgs.columns]

    for label, color in INFRA_CATEGORY_STYLES.items():
        if 'infra_class' not in infra_nodes_wgs.columns:
            continue
        subset = infra_nodes_wgs[infra_nodes_wgs['infra_class'] == label].copy()
        if len(subset) == 0:
            continue
        subset.explore(
            m=m,
            color=color,
            markersize=7,
            tooltip=infra_tooltip_cols,
            popup=infra_popup_cols,
            popup_kwds={'labels': True}
        )

add_infra_legend(m, INFRA_CATEGORY_STYLES)
m

## 9. Interactive map: boundary + rectangle + grid labels + nodes


In [12]:
import folium
from branca.element import MacroElement, Template

grid_labels = rect_grid_gdf.copy()
grid_labels['label_pt'] = grid_labels.geometry.representative_point()

def add_infra_legend(m, color_map, title='Infrastructure nodes'):
    legend_items = ''.join([
        f"<div style='display:flex; align-items:center; margin-bottom:4px;'>"
        f"<span style='display:inline-block; width:12px; height:12px; background:{color}; "
        f"border:1px solid #333; margin-right:8px;'></span>{label}</div>"
        for label, color in color_map.items()
    ])
    html = f"""
    {{% macro html(this, kwargs) %}}
    <div style="
        position: fixed;
        bottom: 40px;
        left: 40px;
        width: 190px;
        z-index: 9999;
        background: white;
        border: 2px solid rgba(0,0,0,0.2);
        border-radius: 6px;
        padding: 10px 12px;
        font-size: 12px;
        box-shadow: 0 1px 5px rgba(0,0,0,0.3);
    ">
        <div style="font-weight: bold; margin-bottom: 8px;">{title}</div>
        <div style='display:flex; align-items:center; margin-bottom:6px;'>
            <span style='display:inline-block; width:12px; height:12px; background:red; border:1px solid #333; margin-right:8px;'></span>Core candidate node
        </div>
        {legend_items}
    </div>
    {{% endmacro %}}
    """
    macro = MacroElement()
    macro._template = Template(html)
    m.get_root().add_child(macro)
    return m

# Prefer the grid-coded layer for visualization when available
if 'grid_coded_nodes' in globals():
    display_nodes_metric = grid_coded_nodes.copy()
else:
    display_nodes_metric = final_nodes_metric.copy()

final_nodes_wgs = display_nodes_metric.to_crs('EPSG:4326').copy()
if 'display_group' in final_nodes_wgs.columns:
    core_nodes_wgs = final_nodes_wgs[final_nodes_wgs['display_group'] != 'infrastructure'].copy()
    infra_nodes_wgs = final_nodes_wgs[final_nodes_wgs['display_group'] == 'infrastructure'].copy()
else:
    core_nodes_wgs = final_nodes_wgs.copy()
    infra_nodes_wgs = final_nodes_wgs.iloc[0:0].copy()

m = boundary_gdf.explore(
    style_kwds={'fillOpacity': 0.02, 'color': 'black', 'weight': 2},
    tooltip=[]
)

adaptive_rect_gdf.explore(
    m=m,
    style_kwds={'fillOpacity': 0.03, 'color': 'green', 'weight': 2},
    tooltip=[]
)

rect_grid_gdf.explore(
    m=m,
    style_kwds={'fillOpacity': 0.03, 'color': 'blue', 'weight': 1},
    tooltip=['grid_id', 'grid_x', 'grid_y']
)

if len(core_nodes_wgs) > 0:
    core_tooltips = [
        col for col in ['grid_node_code', 'grid_x', 'grid_y', 'cell_seq', 'node_type', 'node_subtype']
        if col in core_nodes_wgs.columns
    ]
    core_nodes_wgs.explore(
        m=m,
        color='red',
        markersize=4,
        tooltip=core_tooltips
    )

if len(infra_nodes_wgs) > 0 and 'infra_class' in infra_nodes_wgs.columns:
    for label, color in INFRA_CATEGORY_STYLES.items():
        subset = infra_nodes_wgs[infra_nodes_wgs['infra_class'] == label].copy()
        if len(subset) == 0:
            continue

        for _, row in subset.iterrows():
            popup_fields = []
            for field_name, field_label in [
                ('node_id', 'Node ID'),
                ('grid_node_code', 'Grid code'),
                ('infra_name', 'Name'),
                ('infra_class', 'Class'),
                ('infra_type', 'Type'),
                ('node_subtype', 'Subtype'),
                ('snap_dist_m', 'Snap distance (m)'),
                ('grid_x', 'Grid X'),
                ('grid_y', 'Grid Y'),
            ]:
                if field_name in row.index and pd.notna(row[field_name]):
                    value = row[field_name]
                    if field_name == 'snap_dist_m':
                        value = f"{float(value):.2f}"
                    popup_fields.append(f"<b>{field_label}:</b> {value}")

            popup_html = "<br>".join(popup_fields) if popup_fields else "<b>Infrastructure node</b>"

            folium.CircleMarker(
                location=[row.geometry.y, row.geometry.x],
                radius=5,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.95,
                weight=1,
                tooltip=row.get('infra_name', row.get('node_id', 'Infrastructure node')),
                popup=folium.Popup(popup_html, max_width=320)
            ).add_to(m)

for rec in grid_labels.itertuples():
    label_pt = rec.label_pt
    folium.Marker(
        location=[label_pt.y, label_pt.x],
        icon=folium.DivIcon(
            html=(
                f"<div style='font-size:10px; color:#0B3D91; font-weight:bold;'>"
                f"{rec.grid_id}</div>"
            )
        )
    ).add_to(m)

add_infra_legend(m, INFRA_CATEGORY_STYLES)
m

## 10. Preview grid-coded nodes as a table


In [13]:
preview_source = grid_coded_nodes.copy() if 'grid_coded_nodes' in globals() else final_nodes_metric.copy()
preview = preview_source.to_crs('EPSG:4326').copy()
preview['lat'] = preview.geometry.y
preview['lon'] = preview.geometry.x
cols = [
    col for col in [
        'node_id', 'grid_node_code', 'grid_x', 'grid_y', 'cell_seq',
        'node_type', 'node_subtype', 'display_group', 'infra_name',
        'infra_type', 'notes', 'lat', 'lon'
    ] if col in preview.columns
]
preview[cols].head(30)


,node_id,grid_node_code,grid_x,grid_y,cell_seq,node_type,node_subtype,display_group,infra_name,infra_type,notes,lat,lon
0,N8478,11001,1,1,1,junction,None,core,NaN,NaN,,40.771523,-73.970002
1,N3125,11002,1,1,2,junction,None,core,NaN,NaN,,40.771477,-73.969936
2,N3124,11003,1,1,3,junction,None,core,NaN,NaN,,40.771421,-73.969869
3,N1479,11004,1,1,4,junction,None,core,NaN,NaN,,40.771361,-73.969798
4,INF_0094,11005,1,1,5,infrastructure,amenity:drinking_water,infrastructure,Drinking Water,drinking_water,useful_infrastructure; amenity:drinking_water,40.771056,-73.968783
5,N1489,11006,1,1,6,junction,None,core,NaN,NaN,,40.770988,-73.968713
6,N1484,11007,1,1,7,junction,None,core,NaN,NaN,,40.770604,-73.969326
7,N1466,11008,1,1,8,junction,None,core,NaN,NaN,,40.770392,-73.969860
8,N1441,11009,1,1,9,junction,None,core,NaN,NaN,,40.770184,-73.970025
9,INF_0080,11010,1,1,10,infrastructure,amenity:drinking_water,infrastructure,Drinking Water,drinking_water,useful_infrastructure; amenity:drinking_water,40.770086,-73.970214


## 11. Show exported files


In [14]:
print('central_outputs:')
for p in Path('central_outputs').glob('*'):
    print('  ', p.name)

print('\ncentral_outputs_gridcoded:')
for p in Path('central_outputs_gridcoded').glob('*'):
    print('  ', p.name)


print('\napp_data:')
for p in sorted(Path('app_data').glob('*')):
    print(' -', p.name)

central_outputs:
   augmented_graph_edges.geojson
   base_structural_nodes.geojson
   base_graph_edges.geojson
   infrastructure_nodes_snapped.csv
   final_candidate_nodes.geojson
   final_candidate_nodes.csv
   infrastructure_nodes_snapped.geojson

central_outputs_gridcoded:
   infrastructure_nodes_gridcoded.geojson
   adaptive_rectangle_grid.geojson
   infrastructure_nodes_gridcoded.csv
   adaptive_rectangle.geojson
   final_candidate_nodes_gridcoded.csv
   final_candidate_nodes_gridcoded.geojson

app_data:
 - app_manifest.json
 - augmented_graph_edges.geojson
 - final_candidate_nodes_gridcoded.csv
 - final_candidate_nodes_gridcoded.geojson
 - gate_nodes_gridcoded.csv
 - gate_nodes_gridcoded.geojson
 - infrastructure_nodes_gridcoded.csv
 - infrastructure_nodes_gridcoded.geojson
 - park_graph.pkl


In [15]:
import math
import numpy as np
import geopandas as gpd
import networkx as nx

from scipy.spatial import cKDTree
from shapely.geometry import Point
from pyproj import Transformer

from ipyleaflet import (
    Map, Marker, Polyline, GeoJSON,
    LayersControl, WidgetControl
)
import ipywidgets as widgets


# ----------------------------
# 0. Basic assumptions
# ----------------------------
# Assumes these already exist in your notebook:
# - base_G: nx.Graph with node attr "geometry" in metric CRS
# - metric_crs: CRS string/object for base_G geometries

if "base_G" not in globals():
    raise ValueError("base_G not found. Run your graph-building cell first.")
if "metric_crs" not in globals():
    raise ValueError("metric_crs not found. Run your graph-building cell first.")


# ----------------------------
# 1. Graph -> GeoDataFrames
# ----------------------------
def graph_nodes_to_gdf_for_display(G, crs):
    rows = []
    for nid, attrs in G.nodes(data=True):
        rows.append({
            "node_id": nid,
            "geometry": attrs["geometry"]
        })
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=crs)

def graph_edges_to_gdf_for_display(G, crs):
    rows = []
    for u, v, data in G.edges(data=True):
        rows.append({
            "u": u,
            "v": v,
            "edge_id": data.get("edge_id"),
            "length_m": data.get("length_m", data["geometry"].length),
            "sinuosity": data.get("sinuosity", 1.0),
            "curvature": data.get("curvature", 0.0),
            "geometry": data["geometry"],
        })
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=crs)

nodes_metric = graph_nodes_to_gdf_for_display(base_G, metric_crs)
edges_metric = graph_edges_to_gdf_for_display(base_G, metric_crs)

nodes_wgs = nodes_metric.to_crs("EPSG:4326")
edges_wgs = edges_metric.to_crs("EPSG:4326")


# ----------------------------
# 2. Coordinate transformers
# ----------------------------
to_metric = Transformer.from_crs("EPSG:4326", metric_crs, always_xy=True)
to_wgs = Transformer.from_crs(metric_crs, "EPSG:4326", always_xy=True)


# ----------------------------
# 3. KD-tree for nearest-node snapping
# ----------------------------
node_ids = list(base_G.nodes())
node_xy = np.array([
    [base_G.nodes[n]["geometry"].x, base_G.nodes[n]["geometry"].y]
    for n in node_ids
], dtype=float)

node_tree = cKDTree(node_xy)

def snap_click_to_nearest_node(lat, lon):
    x, y = to_metric.transform(lon, lat)
    dist, idx = node_tree.query([x, y], k=1)
    snapped_node = node_ids[idx]
    snapped_geom = base_G.nodes[snapped_node]["geometry"]
    snapped_lon, snapped_lat = to_wgs.transform(snapped_geom.x, snapped_geom.y)
    return {
        "clicked_lat": lat,
        "clicked_lon": lon,
        "metric_x": x,
        "metric_y": y,
        "node_id": snapped_node,
        "snap_dist_m": float(dist),
        "snap_lat": float(snapped_lat),
        "snap_lon": float(snapped_lon),
    }


# ----------------------------
# 4. Route cost function
# ----------------------------
# This version strongly prefers short distance,
# but adds a penalty for more winding edges.
#
# You can tune alpha / beta later.
alpha_sinuosity = 25.0   # penalty for edges whose sinuosity > 1
beta_curvature = 150.0   # penalty for curved edges

def route_cost(u, v, d):
    length = float(d.get("length_m", d["geometry"].length))
    sinuosity = float(d.get("sinuosity", 1.0))
    curvature = float(d.get("curvature", 0.0))

    sinuosity_penalty = max(0.0, sinuosity - 1.0)
    curvature_penalty = max(0.0, curvature)

    return length + alpha_sinuosity * sinuosity_penalty + beta_curvature * curvature_penalty


# ----------------------------
# 5. A* heuristic
# ----------------------------
# Admissible enough here if your edge cost is nonnegative and
# still fundamentally length-dominated.
def heuristic_metric(n1, n2):
    p1 = base_G.nodes[n1]["geometry"]
    p2 = base_G.nodes[n2]["geometry"]
    return float(p1.distance(p2))


# ----------------------------
# 6. Convert path -> detailed lat/lon polyline
# ----------------------------
def edge_coords_in_path_order(G, u, v):
    line = G[u][v]["geometry"]
    coords = list(line.coords)

    pu = G.nodes[u]["geometry"]
    pv = G.nodes[v]["geometry"]

    start_dist = Point(coords[0]).distance(pu)
    end_dist = Point(coords[-1]).distance(pu)

    # If line is not oriented from u -> v, reverse it
    if start_dist <= end_dist:
        ordered = coords
    else:
        ordered = coords[::-1]

    return ordered

def path_to_latlon_coords(G, path):
    route_coords_metric = []

    for i in range(len(path) - 1):
        u = path[i]
        v = path[i + 1]
        edge_coords = edge_coords_in_path_order(G, u, v)

        # avoid duplicating shared vertices
        if i > 0:
            edge_coords = edge_coords[1:]

        route_coords_metric.extend(edge_coords)

    route_latlon = []
    for x, y in route_coords_metric:
        lon, lat = to_wgs.transform(x, y)
        route_latlon.append((lat, lon))

    return route_latlon

def route_stats(G, path):
    total_length = 0.0
    total_cost = 0.0
    total_sinuosity_penalty = 0.0
    total_curvature_penalty = 0.0

    for i in range(len(path) - 1):
        u, v = path[i], path[i + 1]
        d = G[u][v]

        length = float(d.get("length_m", d["geometry"].length))
        sinuosity = float(d.get("sinuosity", 1.0))
        curvature = float(d.get("curvature", 0.0))

        total_length += length
        total_cost += route_cost(u, v, d)
        total_sinuosity_penalty += max(0.0, sinuosity - 1.0)
        total_curvature_penalty += max(0.0, curvature)

    return {
        "length_m": total_length,
        "cost": total_cost,
        "sinuosity_penalty_sum": total_sinuosity_penalty,
        "curvature_penalty_sum": total_curvature_penalty,
        "num_nodes": len(path),
    }


# ----------------------------
# 7. Optional display layers
# ----------------------------
edge_layer = GeoJSON(
    data=edges_wgs.__geo_interface__,
    name="Walkable edges",
    style={
        "color": "#4C78A8",
        "weight": 2,
        "opacity": 0.6,
    }
)

# Only show candidate/final nodes if they already exist in notebook
candidate_layers = []
if "final_nodes_metric" in globals():
    final_nodes_wgs = final_nodes_metric.to_crs("EPSG:4326")
    if "display_group" in final_nodes_wgs.columns:
        core_nodes_wgs = final_nodes_wgs[final_nodes_wgs["display_group"] != "infrastructure"].copy()
        infra_nodes_wgs = final_nodes_wgs[final_nodes_wgs["display_group"] == "infrastructure"].copy()
    else:
        core_nodes_wgs = final_nodes_wgs.copy()
        infra_nodes_wgs = final_nodes_wgs.iloc[0:0].copy()

    if len(core_nodes_wgs) > 0:
        candidate_layers.append(
            GeoJSON(
                data=core_nodes_wgs.__geo_interface__,
                name="Candidate nodes",
                point_style={
                    "radius": 4,
                    "color": "#E45756",
                    "fillColor": "#E45756",
                    "fillOpacity": 0.8,
                }
            )
        )

    if len(infra_nodes_wgs) > 0:
        candidate_layers.append(
            GeoJSON(
                data=infra_nodes_wgs.__geo_interface__,
                name="Infrastructure nodes",
                point_style={
                    "radius": 5,
                    "color": "#2E8B57",
                    "fillColor": "#2E8B57",
                    "fillOpacity": 0.9,
                }
            )
        )

In [16]:
from IPython.display import display

# ----------------------------
# 1. Map center
# ----------------------------
minx, miny, maxx, maxy = edges_wgs.total_bounds
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

m = Map(center=(center_lat, center_lon), zoom=15)

m.add(edge_layer)
for layer in candidate_layers:
    m.add(layer)

m.add(LayersControl(position="topright"))


# ----------------------------
# 2. UI widgets
# ----------------------------
status_html = widgets.HTML(
    value="<b>Instructions:</b> click once for start, click again for end. A third click starts a new route."
)
route_info_html = widgets.HTML(value="")
reset_btn = widgets.Button(description="Reset route", button_style="warning")

control_box = widgets.VBox([status_html, route_info_html, reset_btn])
m.add(WidgetControl(widget=control_box, position="topright"))


# ----------------------------
# 3. State
# ----------------------------
state = {
    "start": None,
    "end": None,
    "start_marker": None,
    "end_marker": None,
    "route_layer": None,
}

def remove_layer_safe(layer):
    if layer is not None and layer in m.layers:
        m.remove(layer)

def clear_route():
    remove_layer_safe(state["start_marker"])
    remove_layer_safe(state["end_marker"])
    remove_layer_safe(state["route_layer"])

    state["start"] = None
    state["end"] = None
    state["start_marker"] = None
    state["end_marker"] = None
    state["route_layer"] = None

    route_info_html.value = ""
    status_html.value = "<b>Instructions:</b> click once for start, click again for end. A third click starts a new route."

reset_btn.on_click(lambda btn: clear_route())


# ----------------------------
# 4. Route computation
# ----------------------------
def compute_and_draw_route():
    try:
        source = state["start"]["node_id"]
        target = state["end"]["node_id"]

        path = nx.astar_path(
            base_G,
            source=source,
            target=target,
            heuristic=heuristic_metric,
            weight=route_cost,
        )

        state["path"] = path

       
        route_latlon = path_to_latlon_coords(base_G, path)

        route_coords_metric = []

        total_length = 0.0
        total_cost = 0.0
        total_sinuosity_penalty = 0.0
        total_curvature_penalty = 0.0

        for i in range(len(path) - 1):
            u, v = path[i], path[i + 1]
            d = base_G[u][v]

            length = float(d.get("length_m", d["geometry"].length))
            sinuosity = float(d.get("sinuosity", 1.0))
            curvature = float(d.get("curvature", 0.0))

            total_length += length
            total_cost += route_cost(u, v, d)
            total_sinuosity_penalty += max(0.0, sinuosity - 1.0)
            total_curvature_penalty += max(0.0, curvature)

            edge_coords = edge_coords_in_path_order(base_G, u, v)
            if i > 0:
                edge_coords = edge_coords[1:]
            route_coords_metric.extend(edge_coords)

        route_line_metric = LineString(route_coords_metric)
        candidate_count = None
        hit_gdf = None

        if "final_nodes_metric" in globals():
            candidates_metric = final_nodes_metric.to_crs(metric_crs).copy()
            snap_tolerance_m = 8.0

            hit_mask = candidates_metric.geometry.distance(route_line_metric) <= snap_tolerance_m
            hit_gdf = candidates_metric[hit_mask].copy()
            candidate_count = len(hit_gdf)

        state["candidate_nodes_on_path_gdf"] = hit_gdf

        route_line = Polyline(
            locations=route_latlon,
            color="red",
            weight=5,
            fill=False,
            name="A* route"
        )

        remove_layer_safe(state["route_layer"])
        state["route_layer"] = route_line
        m.add(route_line)

        route_info_html.value = (
            f"<b>Route found</b><br>"
            f"start node: {source}<br>"
            f"end node: {target}<br>"
            f"route length: {total_length:.1f} m<br>"
            f"route cost: {total_cost:.1f}<br>"
            f"candidate nodes on path: {candidate_count}<br>"
            f"sinuosity penalty sum: {total_sinuosity_penalty:.3f}<br>"
            f"curvature penalty sum: {total_curvature_penalty:.3f}"
        )

        status_html.value = "<b>Done:</b> click again anywhere to start a new route."

    except nx.NetworkXNoPath:
        route_info_html.value = "<b>No path found.</b>"
        status_html.value = "<b>Try again:</b> click anywhere to start a new route."

# ----------------------------
# 5. Click handler
# ----------------------------
def handle_map_click(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs.get("coordinates")
    snapped = snap_click_to_nearest_node(lat, lon)

    # third click starts over
    if state["start"] is not None and state["end"] is not None:
        clear_route()

    marker = Marker(location=(snapped["snap_lat"], snapped["snap_lon"]))

    if state["start"] is None:
        state["start"] = snapped
        state["start_marker"] = marker
        m.add(marker)

        status_html.value = (
            f"<b>Start set</b><br>"
            f"clicked: ({lat:.6f}, {lon:.6f})<br>"
            f"snapped to node: {snapped['node_id']}<br>"
            f"snap distance: {snapped['snap_dist_m']:.1f} m<br>"
            f"Now click the destination."
        )

    elif state["end"] is None:
        state["end"] = snapped
        state["end_marker"] = marker
        m.add(marker)

        status_html.value = (
            f"<b>End set</b><br>"
            f"clicked: ({lat:.6f}, {lon:.6f})<br>"
            f"snapped to node: {snapped['node_id']}<br>"
            f"snap distance: {snapped['snap_dist_m']:.1f} m<br>"
            f"Computing A* route..."
        )

        compute_and_draw_route()

m.on_interaction(handle_map_click)

display(m)

Map(center=[40.782499099999995, -73.96564624999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…